# 09 — Equal-weight backtest and full comparison


In [2]:
from spy_alpha_research.data import (
    load_config, project_root, ensure_spy, save_frame, load_frame,
    price_series, compute_returns, make_target, split_by_dates,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]

from spy_alpha_research.attribution import load_alpha_candidates
from spy_alpha_research.alpha import build_alpha_signals, equal_weight_blend
from spy_alpha_research.reporting import compare_signals, plot_equity_curves, plot_comparison_table_bars
from spy_alpha_research.backtesting import run_timing_backtest
feats = load_frame(PROC / "features_raw.parquet")
ret = load_frame(PROC / "spy_returns.parquet").iloc[:, 0]
target = load_frame(PROC / "spy_target.parquet").iloc[:, 0]
train_idx, test_idx = split_by_dates(
    feats.index, exp["dates"]["train_end"], exp["dates"]["test_start"], exp["dates"]["end"]
)
alphas = load_alpha_candidates(ROOT / exp["paths"]["alpha_candidates"])
signals = build_alpha_signals(
    feats, alphas, target, train_idx, min_periods=exp["scaling"]["min_periods"]
)
signals["eqw_alpha"] = equal_weight_blend(signals)
hold = {k: v.reindex(test_idx) for k, v in signals.items()}
cmp = compare_signals(
    hold, ret.reindex(test_idx), target.reindex(test_idx),
    signal_lag=bt["signal_lag"], position=bt["position"], costs_bps=bt["costs_bps"],
    periods_per_year=bt["periods_per_year"],
    ic_window=exp["statistics"]["rolling_ic_window"], nw_lags=exp["alpha"]["nw_lags"],
)
curves = {
    k: run_timing_backtest(
        v, ret.reindex(test_idx),
        signal_lag=bt["signal_lag"], position=bt["position"], costs_bps=bt["costs_bps"],
    ).net_returns
    for k, v in hold.items()
}
plot_equity_curves(curves, FIG / "eqw_vs_singles_equity.png", title="Holdout net equity")
plot_comparison_table_bars(cmp, "sharpe", FIG / "all_signals_compare.png")
cmp


,ic,rolling_ic_mean,t_stat,p_value,n_rolling,ann_return,ann_vol,sharpe,max_drawdown,hit_rate,avg_turnover
signal,,,,,,,,,,,
dow,0.009973,-0.019101,-1.724615,8.504608e-02,689,-0.013560,0.113308,-0.063955,-0.242408,0.462766,0.725063
hl_range,-0.002136,-0.076792,-8.799211,1.103486e-17,689,-0.046293,0.103631,-0.405584,-0.184977,0.518617,0.367423
mom_10,0.028906,0.047884,6.375344,3.346508e-10,689,-0.012316,0.115779,-0.049234,-0.153005,0.474734,0.201635
mom_126,-0.003468,0.122929,14.441706,1.738806e-41,689,-0.032083,0.133157,-0.178334,-0.180216,0.442819,0.071986
mom_21,0.005653,0.075672,8.409725,2.366022e-16,689,-0.022421,0.119871,-0.129227,-0.160413,0.464096,0.140873
mom_63,0.046746,0.164930,13.239055,8.361763e-36,689,0.041812,0.124533,0.391051,-0.159409,0.457447,0.095405
month,-0.005934,0.008458,0.724181,4.692008e-01,689,-0.020938,0.116580,-0.123184,-0.152519,0.498670,0.013661
ret_lag_1,-0.048549,-0.066400,-4.816714,1.796169e-06,689,-0.116159,0.109128,-1.076700,-0.380200,0.469415,0.639108
eqw_alpha,0.010355,0.048040,3.773750,1.746780e-04,689,-0.002010,0.069404,0.005643,-0.097690,0.438830,0.195679
